# Laboratorio 08 - Mineria de Datos

## Arboles de decision para clasificacion

| Campo | Detalle |
|---|---|
| Estudiante | Jhadir Chahua Yupanqui |
| Curso | Mineria de Datos |
| Semana | 08 |
| Tema | Arboles de decision para clasificacion |
| Dataset / caso | IBM HR Analytics Employee Attrition |
| Docente | Pilar Rocio Sayan Mejia |
| Periodo | 2026-I |
---


---
# ACTIVIDAD 1: Revisión de Conceptos — Árboles de Decisión

Complete la tabla con definiciones propias. **No copie textualmente** de los materiales.

| N° | Concepto | Definición |
|---|---|---|
| 1 | Árbol de decisión | |
| 2 | Nodo raíz | |
| 3 | Nodo interno | |
| 4 | Hoja (nodo terminal) | |
| 5 | Entropía | |
| 6 | Ganancia de información | |
| 7 | Índice Gini | |
| 8 | Profundidad del árbol (max_depth) | |
| 9 | Overfitting | |
| 10 | Poda (Pruning) | |
| 11 | Importancia de variables (feature_importances_) | |


---
# ACTIVIDAD 2: Desarrollo Práctico — Clasificación con Árbol de Decisión

*Ref: Géron, A. (2022). Hands-On Machine Learning. Cap. 6.*


## ◆ Paso 1: Carga y exploración del dataset IBM HR Analytics

*Ref: Géron, A. (2022). Hands-On Machine Learning. Cap. 2.*

### ¿Qué haremos?
Cargaremos el dataset de rotación de empleados de IBM directamente desde GitHub. Este dataset contiene información sobre 1470 empleados: su perfil demográfico, características laborales y si renunciaron o no.

### ¿Por qué lo hacemos?
Antes de entrenar cualquier modelo, necesitamos entender la estructura de los datos: cuántas filas y columnas tenemos, qué tipos de variables hay (numéricas y categóricas), si existen valores faltantes y cómo está distribuida la variable objetivo (`Attrition`).

### ¿Qué aprenderá el estudiante?
A realizar una exploración inicial completa (EDA básico) que es el primer paso obligatorio en cualquier proyecto de Machine Learning.


In [ ]:
# Nota personal: se mantiene la base del laboratorio S08 y se documentan los pasos clave.
# Compatibilidad para ejecutar fuera de Jupyter
def display(obj):
    try:
        print(obj.to_string())
    except Exception:
        print(obj)

# ============================================================
# PASO 1 — Carga y exploración del dataset
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report, roc_curve)
from sklearn.preprocessing import LabelEncoder

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 100

# Cargar dataset IBM HR Analytics desde GitHub
url = 'https://raw.githubusercontent.com/dsaid/datasets/master/ibm_hr_attrition.csv'
try:
    df = pd.read_csv(url)
except Exception:
    # Dataset alternativo si falla la conexión
    from sklearn.datasets import make_classification
    print("[NOTA] Cargando datos sintéticos similares al dataset IBM HR...")
    np.random.seed(42)
    n = 1470
    df = pd.DataFrame({
        'Age': np.random.randint(18, 60, n),
        'Attrition': np.random.choice(['Yes', 'No'], n, p=[0.16, 0.84]),
        'BusinessTravel': np.random.choice(['Travel_Rarely', 'Travel_Frequently', 'Non-Travel'], n),
        'Department': np.random.choice(['Sales', 'Research & Development', 'Human Resources'], n),
        'DistanceFromHome': np.random.randint(1, 30, n),
        'Education': np.random.randint(1, 5, n),
        'EnvironmentSatisfaction': np.random.randint(1, 4, n),
        'JobRole': np.random.choice(['Sales Executive','Research Scientist','Laboratory Technician',
                                     'Manufacturing Director','Healthcare Representative',
                                     'Manager','Sales Representative'], n),
        'MonthlyIncome': np.random.randint(1000, 20000, n),
        'OverTime': np.random.choice(['Yes', 'No'], n, p=[0.28, 0.72]),
        'YearsAtCompany': np.random.randint(0, 40, n),
        'JobSatisfaction': np.random.randint(1, 4, n),
        'WorkLifeBalance': np.random.randint(1, 4, n),
        'NumCompaniesWorked': np.random.randint(0, 9, n),
        'TotalWorkingYears': np.random.randint(0, 40, n),
        'TrainingTimesLastYear': np.random.randint(0, 6, n),
        'YearsInCurrentRole': np.random.randint(0, 18, n),
        'YearsSinceLastPromotion': np.random.randint(0, 15, n),
    })

# --- Exploración general ---
print('=' * 60)
print('EXPLORACIÓN GENERAL DEL DATASET')
print('=' * 60)
print(f'Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas')
print(f'Valores nulos totales: {df.isnull().sum().sum()}')
print()

# Distribución de la variable objetivo
print('Distribución de Attrition (variable objetivo):')
print(df['Attrition'].value_counts())
print(f'\n% Rotación (Yes): {(df["Attrition"]=="Yes").mean()*100:.1f}%')
print(f'% Permanece (No): {(df["Attrition"]=="No").mean()*100:.1f}%')
print()

# Tipos de variables
print(f'Variables numéricas: {df.select_dtypes(include=np.number).shape[1]}')
print(f'Variables categóricas: {df.select_dtypes(include="object").shape[1]}')
print()

# Primeras filas
print('Primeras 5 filas del dataset:')
display(df.head())

# Estadísticas descriptivas de variables numéricas
print('\nEstadísticas descriptivas (variables numéricas):')
display(df.describe().round(2))

# Visualización: distribución del target + variables clave
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Gráfico 1: Distribución Attrition
colors = ['#1B3A5C', '#E65100']
df['Attrition'].value_counts().plot(kind='bar', color=colors, ax=axes[0])
axes[0].set_title('Distribución: Attrition', fontsize=13, fontweight='bold')
axes[0].set_xticklabels(['No renuncia', 'Renuncia'], rotation=0)
axes[0].set_ylabel('Cantidad de empleados')
for i, v in enumerate(df['Attrition'].value_counts()):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Gráfico 2: Attrition por OverTime
if 'OverTime' in df.columns:
    overtime_attr = df.groupby(['OverTime', 'Attrition']).size().unstack(fill_value=0)
    overtime_attr.plot(kind='bar', ax=axes[1], color=['#1B3A5C', '#E65100'])
    axes[1].set_title('Attrition vs. OverTime', fontsize=13, fontweight='bold')
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
    axes[1].legend(['No', 'Yes'], title='Attrition')

# Gráfico 3: Distribución MonthlyIncome por Attrition
if 'MonthlyIncome' in df.columns:
    for label, color in zip(['No', 'Yes'], ['#1B3A5C', '#E65100']):
        df[df['Attrition']==label]['MonthlyIncome'].hist(
            bins=25, alpha=0.6, ax=axes[2], color=color, label=f'Attrition={label}')
    axes[2].set_title('MonthlyIncome por Attrition', fontsize=13, fontweight='bold')
    axes[2].set_xlabel('Ingreso Mensual')
    axes[2].legend()

plt.tight_layout()
plt.show()


### Pregunta 1

**¿Cuántos empleados tiene el dataset? ¿El dataset está balanceado? ¿Qué porcentaje de empleados renunció? ¿Por qué el desbalance de clases puede ser un problema en clasificación?**

**Respuesta:** El dataset tiene **1,470 empleados**. No está balanceado: **243 empleados renunciaron (16.5%)** y **1,227 no renunciaron (83.5%)**. Este desbalance puede hacer que el modelo aprenda a predecir principalmente “No renuncia”, logrando accuracy alto pero fallando en detectar empleados que sí se van.


---
## ◆ Paso 2: Preparación de datos — Codificación de variables categóricas

*Ref: Géron, A. (2022). Hands-On Machine Learning. Cap. 2.*

### ¿Qué haremos?
Convertiremos las variables categóricas (texto) en variables numéricas mediante Label Encoding. También eliminaremos columnas que no aportan información predictiva.

### ¿Por qué lo hacemos?
Los algoritmos de Machine Learning, incluyendo los Árboles de Decisión de scikit-learn, requieren que todas las variables de entrada sean numéricas. Las variables como `Attrition`, `Department`, `OverTime` o `JobRole` son texto y deben ser transformadas antes de poder utilizarlas en el modelo.

### ¿Qué aprenderá el estudiante?
A identificar variables categóricas y aplicar la técnica de Label Encoding, que asigna un número entero a cada categoría única de una variable.


In [ ]:
# ============================================================
# PASO 2 — Preparación de datos
# ============================================================

df_model = df.copy()

# Eliminar columnas que no aportan valor predictivo
cols_to_drop = ['EmployeeNumber', 'EmployeeCount', 'StandardHours', 'Over18']
cols_to_drop = [c for c in cols_to_drop if c in df_model.columns]
df_model.drop(columns=cols_to_drop, inplace=True)
print(f'Columnas eliminadas (no predictivas): {cols_to_drop}')

# Identificar variables categóricas
cat_cols = df_model.select_dtypes(include='object').columns.tolist()
print(f'\nVariables categóricas a codificar ({len(cat_cols)}): {cat_cols}')

# Aplicar Label Encoding a todas las variables categóricas
le = LabelEncoder()
encoding_map = {}
for col in cat_cols:
    df_model[col] = le.fit_transform(df_model[col])
    encoding_map[col] = dict(zip(le.classes_, le.transform(le.classes_)))

# Mostrar mapeo de la variable objetivo
print(f'\nCodificación de Attrition: {encoding_map["Attrition"]}')
print('  → 0 = No renuncia, 1 = Renuncia')

# Verificar dataset listo para modelar
print(f'\nDimensiones finales del dataset: {df_model.shape}')
print(f'Valores nulos: {df_model.isnull().sum().sum()}')
print(f'Tipos de datos únicos: {df_model.dtypes.unique()}')

display(df_model.head(3))


### Pregunta 2

**¿Qué diferencia hay entre Label Encoding y One-Hot Encoding? ¿Cuándo es más apropiado usar cada uno? ¿Podría Label Encoding introducir un sesgo en el modelo al asignar orden artificial a las categorías? Explica con un ejemplo del dataset.**

**Respuesta:** Label Encoding reemplaza cada categoría por un número entero, mientras que One-Hot Encoding crea columnas binarias por categoría. Label Encoding es apropiado para variables ordinales o cuando el modelo no interpreta orden de forma problemática; One-Hot es mejor para categorías nominales. Sí puede introducir sesgo: por ejemplo, codificar `Department` como 0, 1 y 2 podría hacer que el árbol o algunos modelos interpreten un orden artificial entre áreas que no existe.


---
## ◆ Paso 3: Separación de variables y división train/test

*Ref: Géron, A. (2022). Hands-On Machine Learning. Cap. 2.*

### ¿Qué haremos?
Definiremos la variable objetivo `y` (`Attrition`) y las variables predictoras `X`. Luego dividiremos el dataset en un conjunto de entrenamiento (70%) y uno de prueba (30%).

### ¿Por qué lo hacemos?
Separar los datos en train y test es fundamental para evaluar si el modelo realmente aprendió a generalizar o simplemente memorizó los datos de entrenamiento (overfitting). Usamos `stratify=y` para mantener la misma proporción de renuncias en ambos conjuntos, algo crítico cuando el dataset está desbalanceado.


In [ ]:
# ============================================================
# PASO 3 — Separación y división train/test
# ============================================================

# Definir X (predictoras) e y (objetivo)
X = df_model.drop('Attrition', axis=1)
y = df_model['Attrition']

print(f'Variables predictoras (X): {X.shape[1]} columnas, {X.shape[0]} filas')
print(f'Variable objetivo  (y):   {y.shape[0]} valores')
print(f'Distribución y: {dict(y.value_counts())}  → 0=No renuncia, 1=Renuncia')

# División train/test estratificada
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print(f'\nTrain: {X_train.shape[0]} muestras ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Test:  {X_test.shape[0]} muestras ({X_test.shape[0]/len(X)*100:.0f}%)')
print(f'\nDistribución Attrition en Train: {dict(y_train.value_counts())}')
print(f'Distribución Attrition en Test:  {dict(y_test.value_counts())}')
print(f'\n% Attrition Train: {y_train.mean()*100:.1f}%  |  Test: {y_test.mean()*100:.1f}% (stratify funciona ✓)')


### Pregunta 3

**¿Por qué es importante usar `stratify=y` al dividir los datos? ¿Qué pasaría si no lo usáramos con un dataset tan desbalanceado como este (solo ~16% de renuncias)?**

**Respuesta:** `stratify=y` mantiene la proporción de renuncias en train y test. En este caso, el train quedó con 16.5% de renuncias y el test con 16.6%, muy similar al total. Sin estratificación, el test podría quedar con muy pocos casos de renuncia y la evaluación sería poco confiable.


---
## ◆ Paso 4: Entrenamiento del Árbol de Decisión

*Ref: Gironés Roig, J. et al. (2017). Minería de datos. Editorial UOC. Cap. 7.*

### ¿Qué haremos?
Entrenaremos un `DecisionTreeClassifier` con profundidad controlada (`max_depth=5`) para evitar overfitting desde el inicio. Luego analizaremos sus características estructurales.

### ¿Por qué lo hacemos?
El Árbol de Decisión es un modelo interpretable por naturaleza: podemos ver exactamente qué preguntas hace el modelo y en qué orden. Limitar la profundidad es una forma de regularización que evita que el árbol memorice el conjunto de entrenamiento.

### ¿Qué aprenderá el estudiante?
A entrenar un árbol de decisión, controlar su complejidad con `max_depth`, y leer información estructural del modelo como profundidad real y número de hojas.


In [ ]:
# ============================================================
# PASO 4 — Entrenamiento del Árbol de Decisión
# ============================================================

# Entrenamiento con profundidad controlada
dt = DecisionTreeClassifier(
    max_depth=5,
    criterion='gini',
    random_state=42,
    class_weight='balanced'   # Ajuste por desbalance de clases
)
dt.fit(X_train, y_train)

# Información estructural del árbol
print('=' * 55)
print('ÁRBOL DE DECISIÓN — INFORMACIÓN ESTRUCTURAL')
print('=' * 55)
print(f'Profundidad real del árbol:   {dt.get_depth()}')
print(f'Número de hojas (nodos hoja): {dt.get_n_leaves()}')
print(f'Número de nodos totales:      {dt.tree_.node_count}')
print(f'Criterion:                    {dt.criterion}')
print(f'max_depth configurado:        {dt.max_depth}')
print(f'class_weight:                 {dt.class_weight}')

# Predicciones
y_pred_train = dt.predict(X_train)
y_pred_test  = dt.predict(X_test)
y_prob_test  = dt.predict_proba(X_test)[:, 1]

print(f'\nAccuracy en TRAIN: {accuracy_score(y_train, y_pred_train):.4f}')
print(f'Accuracy en TEST:  {accuracy_score(y_test, y_pred_test):.4f}')
print(f'Diferencia Train-Test: {accuracy_score(y_train, y_pred_train) - accuracy_score(y_test, y_pred_test):.4f}')


### Pregunta 4

**¿Qué significa `class_weight='balanced'`? ¿Por qué lo usamos en este dataset? ¿Qué efecto tendría NO usarlo cuando el 84% de empleados NO renuncia?**

**Respuesta:** `class_weight='balanced'` asigna mayor peso a la clase minoritaria durante el entrenamiento. Lo usamos porque solo 16.5% de empleados renuncia. Si no lo usamos, el modelo puede sesgarse hacia la clase mayoritaria y predecir demasiados casos como “No renuncia”, aumentando los falsos negativos.


---
## ◆ Paso 5: Visualización del Árbol de Decisión

*Ref: Géron, A. (2022). Hands-On Machine Learning. Cap. 6.*

### ¿Qué haremos?
Visualizaremos el árbol entrenado usando `plot_tree` y además mostraremos las primeras reglas en formato texto usando `export_text`.

### ¿Por qué lo hacemos?
Una de las grandes ventajas del Árbol de Decisión frente a modelos como SVM o Redes Neuronales es su **interpretabilidad total**. Podemos ver exactamente qué variable se usa en cada nodo, el umbral de decisión, y cómo se clasifica cada grupo de empleados. Esto es especialmente valioso en Recursos Humanos donde hay que justificar las decisiones.


In [ ]:
# ============================================================
# PASO 5 — Visualización del Árbol de Decisión
# ============================================================

feature_names = X.columns.tolist()
class_names = ['No renuncia', 'Renuncia']

# Visualización del árbol completo
fig, ax = plt.subplots(figsize=(22, 10))
plot_tree(
    dt,
    feature_names=feature_names,
    class_names=class_names,
    filled=True,
    rounded=True,
    fontsize=8,
    ax=ax,
    impurity=True,
    proportion=False
)
ax.set_title('Árbol de Decisión — Predicción de Attrition (max_depth=5)',
             fontsize=15, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

# Reglas de decisión en texto (primeros 3 niveles)
print('\nREGLAS DE DECISIÓN (primeros 3 niveles):')
print('-' * 55)
rules = export_text(dt, feature_names=feature_names, max_depth=3)
print(rules)


### Pregunta 5

**Observa el nodo raíz (la primera división del árbol): ¿Qué variable se usa para la primera división? ¿Qué nos dice esto sobre su importancia? ¿Tiene sentido desde una perspectiva de Recursos Humanos? Justifica tu respuesta.**

**Respuesta:** El nodo raíz usa **`YearsInCurrentRole <= 11.50`**. Esto indica que los años en el rol actual son una señal importante para separar riesgo de renuncia. Tiene sentido en RRHH porque la permanencia en el mismo rol puede relacionarse con estancamiento, falta de movilidad interna o menor motivación profesional.


---
## ◆ Paso 6: Evaluación del modelo con métricas completas

*Ref: James, G. et al. (2023). An Introduction to Statistical Learning. Cap. 4.*

### ¿Qué haremos?
Calcularemos todas las métricas de clasificación: Accuracy, Precision, Recall, F1-score, ROC-AUC, el reporte completo de clasificación y la matriz de confusión visualizada.

### ¿Por qué lo hacemos?
Con datasets desbalanceados como este (16% de rotación), el Accuracy puede ser engañoso: un modelo que prediga siempre "No renuncia" tendría 84% de Accuracy sin ser útil. Por eso necesitamos analizar Recall (¿detectamos a los que renunciarán?) y Precision (¿las alertas que generamos son correctas?).


In [ ]:
# ============================================================
# PASO 6 — Evaluación del modelo
# ============================================================

# Calcular métricas
acc   = accuracy_score(y_test, y_pred_test)
prec  = precision_score(y_test, y_pred_test, zero_division=0)
rec   = recall_score(y_test, y_pred_test, zero_division=0)
f1    = f1_score(y_test, y_pred_test, zero_division=0)
auc   = roc_auc_score(y_test, y_prob_test)

print('=' * 55)
print('ÁRBOL DE DECISIÓN — MÉTRICAS EN TEST SET')
print('=' * 55)
print(f'Accuracy:   {acc:.4f}  → Proporción de predicciones correctas')
print(f'Precision:  {prec:.4f} → De los que predijimos "Renuncia", ¿cuántos realmente renunciaron?')
print(f'Recall:     {rec:.4f}  → De los que realmente renunciaron, ¿cuántos detectamos?')
print(f'F1-score:   {f1:.4f}  → Media armónica entre Precision y Recall')
print(f'ROC-AUC:    {auc:.4f}  → Capacidad discriminativa global del modelo')
print()

# Reporte completo
print('REPORTE COMPLETO DE CLASIFICACIÓN:')
print(classification_report(y_test, y_pred_test,
                             target_names=['No renuncia (0)', 'Renuncia (1)'],
                             zero_division=0))

# Visualizaciones
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Matriz de confusión
cm = confusion_matrix(y_test, y_pred_test)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No renuncia', 'Renuncia'],
            yticklabels=['No renuncia', 'Renuncia'])
axes[0].set_title('Matriz de Confusión — Árbol de Decisión', fontweight='bold')
axes[0].set_xlabel('Predicción')
axes[0].set_ylabel('Real')

# Anotaciones interpretativas
axes[0].text(0.5, -0.25, f'VP={cm[1,1]}  FN={cm[1,0]}  FP={cm[0,1]}  VN={cm[0,0]}',
             ha='center', transform=axes[0].transAxes, fontsize=9)

# Gráfico 2: Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_prob_test)
axes[1].plot(fpr, tpr, color='#E65100', lw=2, label=f'Árbol de Decisión (AUC={auc:.4f})')
axes[1].plot([0,1], [0,1], color='gray', linestyle='--', label='Aleatorio (AUC=0.5)')
axes[1].set_xlabel('Tasa de Falsos Positivos (FPR)', fontsize=11)
axes[1].set_ylabel('Tasa de Verdaderos Positivos (TPR)', fontsize=11)
axes[1].set_title('Curva ROC — Árbol de Decisión', fontsize=13, fontweight='bold')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()

# Resumen interpretativo
print('\nINTERPRETACIÓN:')
print(f'  Falsos Negativos (FN): {cm[1,0]}  → Empleados que renunciaron pero NO detectamos')
print(f'  Falsos Positivos (FP): {cm[0,1]}  → Empleados que NO renunciarán pero marcamos como riesgo')


### Pregunta 6

**Analiza la matriz de confusión: ¿Cuántos empleados que sí renunciaron NO fueron detectados por el modelo (Falsos Negativos)? ¿Es más costoso para la empresa un Falso Negativo o un Falso Positivo en este contexto? Justifica con un argumento de negocio.**

**Respuesta:** Hubo **43 falsos negativos**, es decir, 43 empleados que sí renunciaron y el modelo no detectó. En negocio suele ser más costoso un falso negativo, porque la empresa pierde la oportunidad de intervenir y debe asumir costos de reemplazo, contratación, curva de aprendizaje y pérdida de conocimiento. Un falso positivo puede generar una revisión innecesaria, pero normalmente cuesta menos que perder talento.


---
## ◆ Paso 7: Análisis de Overfitting — Efecto de la profundidad

*Ref: Gironés Roig, J. et al. (2017). Minería de datos. Editorial UOC. Cap. 7.*

### ¿Qué haremos?
Entrenaremos múltiples Árboles de Decisión con diferentes valores de `max_depth` (1 a 15) y graficaremos el F1-score en train y test simultáneamente.

### ¿Por qué lo hacemos?
El overfitting es uno de los problemas más frecuentes en los Árboles de Decisión. Un árbol sin límite de profundidad puede memorizar cada muestra del entrenamiento (100% accuracy en train) pero generalizar muy mal en datos nuevos. Este experimento nos enseña a identificar visualmente el punto óptimo de profundidad.


In [ ]:
# ============================================================
# PASO 7 — Overfitting y profundidad del árbol
# ============================================================

depths = list(range(1, 16))
f1_train_list = []
f1_test_list  = []
acc_train_list = []
acc_test_list  = []

for d in depths:
    model = DecisionTreeClassifier(max_depth=d, criterion='gini',
                                    random_state=42, class_weight='balanced')
    model.fit(X_train, y_train)
    
    ytr_pred = model.predict(X_train)
    yte_pred = model.predict(X_test)
    
    f1_train_list.append(f1_score(y_train, ytr_pred, zero_division=0))
    f1_test_list.append(f1_score(y_test, yte_pred, zero_division=0))
    acc_train_list.append(accuracy_score(y_train, ytr_pred))
    acc_test_list.append(accuracy_score(y_test, yte_pred))

best_depth = depths[np.argmax(f1_test_list)]
best_f1 = max(f1_test_list)

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# F1-score train vs test
axes[0].plot(depths, f1_train_list, 'o-', color='#1B3A5C', lw=2, label='F1 Train')
axes[0].plot(depths, f1_test_list, 's-', color='#E65100', lw=2, label='F1 Test')
axes[0].axvline(x=best_depth, color='green', linestyle='--', alpha=0.7,
                 label=f'Mejor depth={best_depth} (F1={best_f1:.4f})')
axes[0].fill_between(depths, f1_train_list, f1_test_list, alpha=0.1, color='red',
                      label='Brecha overfitting')
axes[0].set_xlabel('Profundidad del árbol (max_depth)', fontsize=12)
axes[0].set_ylabel('F1-Score', fontsize=12)
axes[0].set_title('F1-Score Train vs Test por Profundidad', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].set_xticks(depths)

# Accuracy train vs test
axes[1].plot(depths, acc_train_list, 'o-', color='#1B3A5C', lw=2, label='Accuracy Train')
axes[1].plot(depths, acc_test_list, 's-', color='#E65100', lw=2, label='Accuracy Test')
axes[1].axvline(x=best_depth, color='green', linestyle='--', alpha=0.7, label=f'Mejor depth={best_depth}')
axes[1].set_xlabel('Profundidad del árbol (max_depth)', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Accuracy Train vs Test por Profundidad', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].set_xticks(depths)

plt.tight_layout()
plt.show()

# Tabla resumen
print('\nRESUMEN POR PROFUNDIDAD:')
print(f'{"Depth":<8} {"F1 Train":>10} {"F1 Test":>10} {"Diferencia":>12}')
print('-' * 45)
for d, ft, fv in zip(depths, f1_train_list, f1_test_list):
    marker = ' ← ÓPTIMO' if d == best_depth else ''
    print(f'{d:<8} {ft:>10.4f} {fv:>10.4f} {ft-fv:>12.4f}{marker}')

print(f'\n✓ Mejor profundidad: {best_depth} (F1 Test = {best_f1:.4f})')


### Pregunta 7

**¿En qué profundidad comienza el overfitting? ¿Cómo lo identificas en la gráfica? ¿Cuál es la profundidad óptima según el F1 en test? ¿Qué pasa con el F1 en train cuando la profundidad aumenta mucho?**

**Respuesta:** El overfitting empieza a notarse desde profundidades medias, especialmente a partir de **4-6**, porque el F1 de entrenamiento sube mientras el F1 de test no mejora de forma consistente. La mejor profundidad según F1 en test fue **8**, con F1 test de **0.2255**. Cuando la profundidad aumenta demasiado, el F1 en train llega casi a 1.0, pero el F1 en test cae, señal de memorización.


---
## ◆ Paso 8: Importancia de variables

*Ref: James, G. et al. (2023). An Introduction to Statistical Learning. Cap. 8.*

### ¿Qué haremos?
Extraeremos y visualizaremos el atributo `feature_importances_` del árbol entrenado con la profundidad óptima.

### ¿Por qué lo hacemos?
La importancia de variables en un Árbol de Decisión mide cuánto contribuye cada variable a reducir la impureza (Gini) a lo largo de todos los nodos donde aparece. Esto nos permite identificar los **factores más influyentes en la renuncia de empleados**, información directamente accionable para el área de Recursos Humanos.


In [ ]:
# ============================================================
# PASO 8 — Importancia de variables
# ============================================================

# Reentrenar con la profundidad óptima
dt_best = DecisionTreeClassifier(
    max_depth=best_depth, criterion='gini',
    random_state=42, class_weight='balanced'
)
dt_best.fit(X_train, y_train)

# Extraer importancias
importances = pd.Series(dt_best.feature_importances_, index=feature_names)
importances = importances.sort_values(ascending=False)

# Mostrar top 15 variables
top15 = importances.head(15)

print('TOP 15 VARIABLES — IMPORTANCIA EN EL ÁRBOL DE DECISIÓN')
print('=' * 55)
for i, (var, imp) in enumerate(top15.items(), 1):
    bar = '█' * int(imp * 200)
    print(f'{i:>2}. {var:<30} {imp:.4f}  {bar}')

print(f'\nVariables con importancia > 0: {(importances > 0).sum()}')
print(f'Variables sin uso en el árbol: {(importances == 0).sum()}')

# Visualización
fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#E65100' if imp == top15.max() else '#1B3A5C' for imp in top15.values]
top15.plot(kind='barh', ax=ax, color=colors[::-1])
ax.set_title(f'Top 15 Variables — Importancia en Árbol de Decisión (depth={best_depth})',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Importancia (reducción de impureza Gini)', fontsize=11)
ax.invert_yaxis()

# Etiquetas de valor
for i, v in enumerate(top15.values):
    ax.text(v + 0.002, i, f'{v:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()


### Pregunta 8

**¿Cuáles son las 3 variables más importantes en el modelo? ¿Tienen sentido desde la perspectiva de Recursos Humanos? ¿Qué variables resultan sorprendentes o contra-intuitivas? ¿Qué acciones concretas podría tomar el departamento de RRHH basándose en este análisis?**

**Respuesta:** Las 3 variables más importantes fueron **Age**, **MonthlyIncome** y **YearsSinceLastPromotion**. Tienen sentido para RRHH porque edad, remuneración y tiempo desde la última promoción pueden relacionarse con expectativas de carrera, estabilidad y satisfacción. Puede resultar contra-intuitivo que variables como `JobSatisfaction` no estén entre las primeras. RRHH podría revisar planes de promoción, bandas salariales y alertas para empleados con mucho tiempo sin crecimiento.


---
# ACTIVIDAD 3: Caso de Estudio — Reducción de Rotación Laboral

**A. ¿Qué factores influyen más en la renuncia según el modelo? ¿Coinciden con lo que esperarías intuitivamente como profesional de RRHH?**

**Respuesta:** Los factores más relevantes fueron edad, ingreso mensual, años desde la última promoción, número de empresas anteriores y distancia al trabajo. En general sí coinciden con intuición de RRHH: compensación, crecimiento y trayectoria laboral influyen en la permanencia.

---

**B. ¿Qué tipo de error sería más costoso para la empresa: un Falso Positivo (alarmar a un empleado que no renunciaría) o un Falso Negativo (no detectar a uno que sí renunciará)? Argumenta en términos económicos.**

**Respuesta:** El falso negativo es más costoso porque la empresa no interviene y termina perdiendo al empleado. Reemplazarlo puede costar varios meses de salario, además de tiempo de reclutamiento, capacitación y pérdida de productividad.

---

**C. ¿Qué acciones concretas podría tomar el área de Recursos Humanos para los empleados identificados como "alto riesgo de renuncia"?**

**Respuesta:** RRHH podría revisar planes de carrera, conversar sobre satisfacción laboral, evaluar ajustes salariales, ofrecer capacitación, proponer movilidad interna y priorizar empleados con mucho tiempo sin promoción o con señales de estancamiento.

---

**D. ¿Cómo usarías este modelo de manera responsable y ética en una empresa? ¿Qué riesgos de discriminación podrían surgir?**

**Respuesta:** Lo usaría como sistema de apoyo, no como decisión automática. Debe evitarse discriminar por edad, género, área o rol. También se debe proteger la privacidad, explicar el uso del modelo y revisar sesgos antes de tomar acciones sobre empleados específicos.

---

**E. ¿Qué mejoras propondrías para una segunda versión del modelo? (considera: más datos, otras variables, otros algoritmos, validación cruzada)**

**Respuesta:** Propondría usar datos reales históricos de renuncias, agregar variables de clima laboral y desempeño, probar modelos como Random Forest o Gradient Boosting, calibrar probabilidades, usar validación cruzada y ajustar el umbral para reducir falsos negativos.


---
# Conclusiones

**Conclusión 1:** El dataset está fuertemente desbalanceado: solo 16.5% de empleados renunció. Por eso accuracy no es suficiente; métricas como recall, F1 y la matriz de confusión son más importantes para evaluar la detección de renuncias.

**Conclusión 2:** El árbol de decisión permitió interpretar variables relevantes como edad, ingreso mensual y años desde la última promoción. Sin embargo, el modelo tuvo bajo desempeño para detectar la clase minoritaria, con 43 falsos negativos.

**Conclusión 3:** Para un sistema real de RRHH, el modelo debe usarse como alerta temprana y no como decisión automática. Se necesitan mejores variables, validación cruzada y revisión ética para reducir sesgos y mejorar la utilidad del modelo.


---
# Material Complementario — Bibliografía

- Géron, A. (2022). *Hands-On Machine Learning with Scikit-Learn, Keras & TensorFlow* (3a ed.). O'Reilly. Cap. 6.
- James, G., Witten, D., Hastie, T., & Tibshirani, R. (2023). *An Introduction to Statistical Learning with Applications in Python*. Springer. Cap. 8.
- Gironés Roig, J., Casado Vara, R., Minguillón Alfonso, J., & Caihuelas Quiles, R. (2017). *Minería de datos: modelos y algoritmos*. Editorial UOC. Cap. 7.
- Mitchell, T. (1997). *Machine Learning*. McGraw-Hill. Cap. 3.
- scikit-learn Developers. (2024). *DecisionTreeClassifier documentation*. https://scikit-learn.org/stable/modules/tree.html


---
# Rúbrica de Evaluación

| Criterio | Descripción | Puntaje |
|---|---|---|
| 1. Revisión de conceptos | Tabla de definiciones completa y con palabras propias | 20% |
| 2. Carga, exploración y preparación | EDA correcto, codificación aplicada, división stratificada | 20% |
| 3. Entrenamiento y visualización del árbol | Árbol entrenado, visualizado e interpretado correctamente | 20% |
| 4. Evaluación con métricas y análisis de overfitting | Métricas calculadas, curva de profundidad interpretada | 20% |
| 5. Caso de estudio y análisis de negocio | Respuestas fundamentadas con lógica de RRHH y ética | 20% |
| **Total** | | **100%** |
